<a href="https://colab.research.google.com/github/writetomangamsudheer-stack/ai-mentor-portfolio/blob/main/Day_10_%E2%80%94_Capstone_Sprint_5_Placement_Prep_Crew_(Updated_Turnkey_Walkthrough)led1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "numpy<2.0"
!pip install -q --upgrade crewai litellm google-generativeai google-genai chromadb sentence-transformers crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 51.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xa

In [ ]:
import os, getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API Key: ")

print("API key set")

Enter Gemini API Key: ··········
API key set


In [ ]:
from crewai.tools import tool

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer

import json
import pathlib

In [ ]:
llm = "gemini/gemini-2.5-flash"

print(llm)

gemini/gemini-2.5-flash


In [ ]:
client_db = PersistentClient(path='./chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

@crewai_tool
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

In [10]:
from chromadb import PersistentClient
from sentence_transformers import SentenceTransformer
from crewai.tools import tool

client_db = PersistentClient(path='/content/drive/MyDrive/syllabi_project/chroma_db')

col = client_db.get_or_create_collection('placement_kb')

embed = SentenceTransformer(
    'sentence-transformers/all-MiniLM-L6-v2'
)

@tool("rag_search")
def rag_search(query: str) -> str:
    """
    Search placement knowledge base.
    """

    qv = embed.encode([query]).tolist()

    results = col.query(
        query_embeddings=qv,
        n_results=4
    )

    docs = results['documents'][0]

    return '\n---\n'.join(docs)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
print(rag_search.run(
    "TCS Digital interview process"
))

TCS Digital - Software Engineer: must-haves: Java, DSA, SQL, OOPS. nice-to-haves: Spring Boot, AWS. min CGPA: 7.0. locations: Hyderabad, Bangalore, Chennai. package: 7.0 LPA.
---
Wipro - Project Engineer: must-haves: Programming Fundamentals, DBMS. nice-to-haves: Python, Cloud. min CGPA: 6.0. locations: Bangalore, Pune. package: 3.5 LPA.
---
Microsoft - Software Engineer Intern: must-haves: DSA, OOPS, C#, Problem Solving. nice-to-haves: .NET, Azure. min CGPA: 8.0. locations: Hyderabad, Bangalore. package: 28.0 LPA.
---
Accenture - Associate Software Engineer: must-haves: Java, Python, SQL, Communication. nice-to-haves: AWS, Azure. min CGPA: 6.5. locations: Hyderabad, Bengaluru, Mumbai. package: 4.5 LPA.


In [12]:
researcher = Agent(
    role="Placement Researcher",

    goal="Research company placement requirements for students",

    backstory="You prepare factual placement research notes using RAG search.",

    llm=llm,

    tools=[rag_search],

    verbose=True,

    allow_delegation=False,
)

In [13]:
interviewer = Agent(
    role="Mock Interviewer",

    goal="Generate placement interview questions",

    backstory="You generate technical and HR interview questions.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [14]:
coach = Agent(
    role="Answer Coach",

    goal="Create strong sample answers and preparation guidance",

    backstory="You coach students with strong placement answers.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [15]:
tracker = Agent(
    role="Progress Tracker",

    goal="Create structured student progress summaries",

    backstory="You generate JSON-style student progress summaries.",

    llm=llm,

    verbose=True,

    allow_delegation=False,
)

In [16]:
profiles = [
    {
        "student_id": "S001",
        "name": "Ravi Kumar",
        "branch": "CSE",
        "cgpa": 7.8,
        "skills": ["Python", "Java", "SQL"],
        "target_company": "TCS Digital"
    },

    {
        "student_id": "S002",
        "name": "Sneha Reddy",
        "branch": "ECE",
        "cgpa": 8.1,
        "skills": ["Python", "DBMS"],
        "target_company": "Cognizant"
    },

    {
        "student_id": "S003",
        "name": "Arun Pillai",
        "branch": "IT",
        "cgpa": 8.5,
        "skills": ["Java", "DSA", "OOPs"],
        "target_company": "Amazon"
    }
]

In [17]:
def build_tasks(profile):

    research = Task(
        description=(
            f"Research placement preparation details for "
            f"{profile['target_company']} "
            f"for a {profile['branch']} student."
        ),

        agent=researcher,

        expected_output="Short bullet research notes."
    )

    interview = Task(
        description=(
            f"Generate EXACTLY 10 mock interview questions "
            f"for {profile['name']} targeting "
            f"{profile['target_company']}."
        ),

        agent=interviewer,

        expected_output="Exactly 10 numbered interview questions."
    )

    coaching = Task(
        description=(
            f"Pick question 3 and create strong sample answer "
            f"for {profile['name']}."
        ),

        agent=coach,

        expected_output="One strong answer and 3 preparation tips."
    )

    tracking = Task(
        description=(
            f"Create JSON-style progress summary "
            f"for {profile['student_id']}."
        ),

        agent=tracker,

        expected_output="Valid JSON-style summary."
    )

    return [research, interview, coaching, tracking]

In [18]:
p = profiles[0]

crew = Crew(
    agents=[
        researcher,
        interviewer,
        coach,
        tracker
    ],

    tasks=build_tasks(p),

    process=Process.sequential,

    verbose=True,
)

In [19]:
result = await crew.kickoff_async()

print("\n=== FINAL OUTPUT ===\n")

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d333a1a0-e8fd-4743-8b14-03f0f88ed589                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 318632bb-a38d-427e-8822-6f5702874eac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: rag_search                                                                                               │
│  Args: {'query': 'TCS Digital placement preparation details for CSE student'}                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool rag_search executed with result: TCS Digital - Software Engineer: must-haves: Java, DSA, SQL, OOPS. nice-to-haves: Spring Boot, AWS. min CGPA: 7.0. locations: Hyderabad, Bangalore, Chennai. package: 7.0 LPA.
---
Microsoft - Software ...


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```json                                                                                                        │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "job_role_target": "TCS Digital - Software Engineer",                                                        │
│    "assessment_date": "2023-10-27",                                                                             │
│    "overall_progress_summary": "Initial assessment indicates a strong foundational understanding in a critical  │
│  Java concept (Exception Handling). Further evaluation and development are required across the remaining        │
│  must-have technical skills and soft skills for the TCS Digital role.",                                         │
│    "skills_assessed": {                                                                                         │
│      "Java": {                                                                                                  │
│        "Exception_Handling": {                                                                                  │
│          "status": "Strong",                                                                                    │
│          "details": "Demonstrated comprehensive understanding of `try`, `catch`, `finally`, and `throws` with   │
│  a practical, well-explained code example (File I/O). Articulated the purpose of each keyword, handled          │
│  resource cleanup effectively using `finally`, and showed awareness of checked vs. unchecked exceptions.",      │
│          "aligns_with_job_criteria": true                                                                       │
│        }                                                                                                        │
│      }                                                                                                          │
│    },                                                                                                           │
│    "remaining_skills_to_assess": [                                                                              │
│      "Java (other advanced topics, e.g., Collections, Multithreading)",                                         │
│      "Data Structures and Algorithms (DSA)",                                                                    │
│      "SQL (Advanced Queries, Database Concepts)",                                                               │
│      "Object-Oriented Programming System (OOPS) (Principles, Design Patterns)",                                 │
│      "Project Experience & Problem Solving",                                                                    │
│      "Staying Updated with Modern Technologies (e.g., Spring Boot, AWS)",                                       │
│      "Career Aspirations and Cultural Fit"                                                                      │
│    ],                                                                                                           │
│    "areas_for_further_development": [                                                                           │
│      "Prepare concise and demonstrative code examples f

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Create JSON-style progress summary for S001.                                                             │
│  Agent: Progress Tracker                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== FINAL OUTPUT ===

```json
{
  "student_id": "S001",
  "job_role_target": "TCS Digital - Software Engineer",
  "assessment_date": "2023-10-27",
  "overall_progress_summary": "Initial assessment indicates a strong foundational understanding in a critical Java concept (Exception Handling). Further evaluation and development are required across the remaining must-have technical skills and soft skills for the TCS Digital role.",
  "skills_assessed": {
    "Java": {
      "Exception_Handling": {
        "status": "Strong",
        "details": "Demonstrated comprehensive understanding of `try`, `catch`, `finally`, and `throws` with a practical, well-explained code example (File I/O). Articulated the purpose of each keyword, handled resource cleanup effectively using `finally`, and showed awareness of checked vs. unchecked exceptions.",
        "aligns_with_job_criteria": true
      }
    }
  },
  "remaining_skills_to_assess": [
    "Java (other advanced topics, e.g., Collections, Multithr

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d333a1a0-e8fd-4743-8b14-03f0f88ed589                                                                       │
│  Final Output: ```json                                                                                          │
│  {                                                                                                              │
│    "student_id": "S001",                                                                                        │
│    "job_role_target": "TCS Digital - Software Engineer",                                                        │
│    "assessment_date": "2023-10-27",                                                                             │
│    "overall_progress_summary": "Initial assessment indicates a strong foundational understanding in a critical  │
│  Java concept (Exception Handling). Further evaluation and development are required across the remaining        │
│  must-have technical skills and soft skills for the TCS Digital role.",                                         │
│    "skills_assessed": {                                                                                         │
│      "Java": {                                                                                                  │
│        "Exception_Handling": {                                                                                  │
│          "status": "Strong",                                                                                    │
│          "details": "Demonstrated comprehensive understanding of `try`, `catch`, `finally`, and `throws` with   │
│  a practical, well-explained code example (File I/O). Articulated the purpose of each keyword, handled          │
│  resource cleanup effectively using `finally`, and showed awareness of checked vs. unchecked exceptions.",      │
│          "aligns_with_job_criteria": true                                                                       │
│        }                                                                                                        │
│      }                                                                                                          │
│    },                                                                                                           │
│    "remaining_skills_to_assess": [                                                                              │
│      "Java (other advanced topics, e.g., Collections, Multithreading)",                                         │
│      "Data Structures and Algorithms (DSA)",                                                                    │
│      "SQL (Advanced Queries, Database Concepts)",                                                               │
│      "Object-Oriented Programming System (OOPS) (Principles, Design Patterns)",                                 │
│      "Project Experience & Problem Solving",                                                                    │
│      "Staying Updated with Modern Technologies (e.g., Spring Boot, AWS)",                                       │
│      "Career Aspirations and Cultural Fit"                                                                      │
│    ],                                                                                                           │
│    "areas_for_further_development": [                                                                           │
│      "Prepare concise and demonstrative code examples 

In [22]:
transcripts = []

for p in profiles:

    print("\n" + "="*60)

    print(f"Running for {p['name']} → {p['target_company']}")

    print("="*60)

    crew = Crew(
        agents=[
            researcher,
            interviewer,
            coach,
            tracker
        ],

        tasks=build_tasks(p),

        process=Process.sequential,

        verbose=True,
    )

    result = await crew.kickoff_async()

    transcripts.append({
        "student": p["name"],
        "target": p["target_company"],
        "final_output": str(result),
    })

print("Completed:", len(transcripts))


Running for Ravi Kumar → TCS Digital


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d08d1ae3-8fb0-4588-819d-9fea40082c06                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research placement preparation details for TCS Digital for a CSE student.                                │
│  ID: 38ddd435-2a0a-47cc-9526-e3b0e84f17c4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Placement Researcher                                                                                    │
│                                                                                                                 │
│  Task: Research placement preparation details for TCS Digital for a CSE student.                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Google Gemini API error: 503 - This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.


...

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Pick question 3 and create strong sample answer for Sneha Reddy.                                         │
│  Agent: Answer Coach                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'agent_execution_started' (expected
'crew_kickoff_started')

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: c930e6a0-c977-41f7-9a82-4f0959b49040                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 40.109422896s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'

In [23]:
pathlib.Path(
    "day10_sprint5_transcripts.json"
).write_text(
    json.dumps(transcripts, indent=2)
)

print("Saved transcripts successfully")

Saved transcripts successfully


In [24]:
md = "# Day10 Sprint5 Report\n\n"

for t in transcripts:

    md += f"## {t['student']} → {t['target']}\n\n"

    md += t["final_output"]

    md += "\n\n---\n\n"

pathlib.Path(
    "day10_sprint5_report.md"
).write_text(md)

print("Markdown report created")

Markdown report created


In [25]:
from google.colab import files

files.download("day10_sprint5_transcripts.json")

files.download("day10_sprint5_report.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>